# Notebook 01: GCR Spectrum Module

This notebook demonstrates the GCR flux generation pipeline (`gcr/spectrum.py`):
- Local Interstellar Spectrum (LIS) for all ion species
- Solar modulation via the Gleeson-Axford force-field approximation
- Solar cycle variation across the 11-year cycle
- Per-species spectral shapes (Boschini et al. 2020 HelMod parameterizations)

**Key references:**
- Vos & Potgieter (2015), ApJ 815:119 — proton LIS
- Boschini et al. (2020), ApJS 250:27 — HZE LIS per species
- Gleeson & Axford (1968), ApJ 154:1011 — force-field modulation
- Usoskin et al. — reconstructed phi database (cosmicrays.oulu.fi)

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join('..', ))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import pandas as pd

from gcr.spectrum import (
    lis_proton, lis_helium, lis_heavy,
    force_field_modulation, gcr_total_flux,
    load_usoskin_phi, phi_at_date,
)
from gcr.utils import ION_SPECIES

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Energy grid: 10 MeV/n to 100 GeV/n
E = np.logspace(1, 5, 300)  # MeV/nucleon
print(f'Energy grid: {E[0]:.0f} – {E[-1]/1000:.0f} GeV/n, {len(E)} points')

## 1. Local Interstellar Spectrum (LIS) — all species

The LIS is the GCR flux outside the heliosphere before any solar modulation.
Each species now has a species-specific parameterization (Boschini et al. 2020).

Key difference from abundance-scaled proton LIS:
- **Fe** has α=2.65 vs proton α=3.93 → much flatter spectrum → more high-energy Fe → higher REID

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

species_colors = {'H': '#1f77b4', 'He': '#ff7f0e', 'C': '#2ca02c',
                  'O': '#d62728', 'Si': '#9467bd', 'Fe': '#8c564b'}

# Left: absolute LIS flux × E^2.7 (standard cosmic ray presentation)
ax = axes[0]
for sp_key, sp in ION_SPECIES.items():
    Z, A = sp['Z'], sp['A']
    if Z == 1 and A == 1:
        flux = lis_proton(E)
    elif Z == 2 and A == 4:
        flux = lis_helium(E)
    else:
        flux = lis_heavy(E, Z, A)
    # Multiply by E^2.7 to flatten spectrum for visibility
    ax.loglog(E, flux * E**2.7, color=species_colors[sp_key],
              label=f"{sp['name']} (Z={Z})", linewidth=2)

ax.set_xlabel('Kinetic energy (MeV/nucleon)')
ax.set_ylabel(r'$j \times E^{2.7}$ (cm$^{-2}$ s$^{-1}$ MeV$^{1.7}$/n sr$^{-1}$)')
ax.set_title('Local Interstellar Spectrum × $E^{2.7}$')
ax.legend(fontsize=8, loc='upper right')
ax.set_xlim(10, 1e5)

# Right: flux ratios to proton (should match ACE CRIS at ~1-10 GeV/n)
ax = axes[1]
flux_H = lis_proton(E)
for sp_key, sp in ION_SPECIES.items():
    if sp_key == 'H':
        continue
    Z, A = sp['Z'], sp['A']
    if Z == 2 and A == 4:
        flux = lis_helium(E)
    else:
        flux = lis_heavy(E, Z, A)
    ratio = flux / flux_H
    ax.semilogx(E, ratio, color=species_colors[sp_key],
                label=f"{sp_key} (abundance={sp['abundance']:.4f})", linewidth=2)
    # Mark ACE CRIS abundance at 1 GeV/n
    ax.axhline(sp['abundance'], color=species_colors[sp_key],
               linestyle='--', alpha=0.4, linewidth=1)

ax.set_xlabel('Kinetic energy (MeV/nucleon)')
ax.set_ylabel('Flux / Proton flux')
ax.set_title('Flux Ratios to Proton (dashed = ACE CRIS abundance)')
ax.legend(fontsize=8)
ax.set_xlim(10, 1e5)
ax.set_ylim(0, 0.15)

plt.suptitle('GCR Local Interstellar Spectrum — Boschini et al. (2020) HelMod',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../figures/01_lis_spectra.png', dpi=150, bbox_inches='tight')
plt.show()

# Print spectral index comparison
print('\nSpectral index comparison at 1-10 GeV/n:')
print(f'{"Species":<8} {"Boschini α":<12} {"vs proton α=3.93"}')
print('-' * 38)
hze_params = {'C': 2.77, 'O': 2.80, 'Si': 2.78, 'Fe': 2.65}
for sp, alpha in hze_params.items():
    diff = alpha - 3.93
    print(f'{sp:<8} {alpha:<12.2f} {diff:+.2f} (flatter = more high-E particles)')

## 2. Force-field solar modulation

The Gleeson-Axford (1968) force-field approximation modulates the LIS by shifting particle energies by (Z/A)×φ. Higher φ (solar maximum) → lower flux at all energies, especially below ~1 GeV/n.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: proton flux at different phi values
ax = axes[0]
phi_values = [300, 450, 550, 700, 900, 1200]  # MV
cmap = plt.cm.RdYlBu_r
colors = cmap(np.linspace(0, 1, len(phi_values)))

flux_LIS = lis_proton(E)
ax.loglog(E, flux_LIS, 'k--', linewidth=1.5, label='LIS (unmodulated)', alpha=0.6)
for phi, color in zip(phi_values, colors):
    flux_mod = force_field_modulation(E, phi, lis_proton)
    ax.loglog(E, flux_mod, color=color, linewidth=1.8,
              label=f'φ = {phi} MV')

ax.set_xlabel('Kinetic energy (MeV/nucleon)')
ax.set_ylabel(r'Differential flux (cm$^{-2}$ s$^{-1}$ MeV$^{-1}$ sr$^{-1}$)')
ax.set_title('Proton LIS vs Modulated Spectra')
ax.legend(fontsize=8, loc='upper left')
ax.set_xlim(10, 1e5)
ax.axvline(1000, color='gray', linestyle=':', alpha=0.5, label='1 GeV/n')

# Right: modulation ratio (modulated / LIS) vs energy for different phi
ax = axes[1]
for phi, color in zip(phi_values, colors):
    flux_mod = force_field_modulation(E, phi, lis_proton)
    ratio = flux_mod / np.maximum(flux_LIS, 1e-30)
    ax.semilogx(E, ratio, color=color, linewidth=1.8, label=f'φ = {phi} MV')

ax.axhline(1.0, color='k', linestyle='--', linewidth=1)
ax.set_xlabel('Kinetic energy (MeV/nucleon)')
ax.set_ylabel('Modulated / LIS flux ratio')
ax.set_title('Solar Modulation Suppression Factor')
ax.legend(fontsize=8)
ax.set_xlim(10, 1e5)
ax.set_ylim(0, 1.05)

plt.suptitle('Gleeson-Axford Force-Field Solar Modulation', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../figures/01_solar_modulation.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Solar cycle variation of modulation potential φ(t)

The solar modulation potential φ varies with the 11-year solar cycle.
Solar minimum (φ~400 MV) → higher GCR flux → higher astronaut dose.

In [ ]:
data_dir = os.path.join('..', 'data', 'usoskin')
phi_path = os.path.join(data_dir, 'phi_monthly.csv')
phi_df = load_usoskin_phi(phi_path)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Top: phi vs time
ax = axes[0]
# Show 2000-2030 range
mask = (phi_df.index.year >= 2000) & (phi_df.index.year <= 2026)
phi_plot = phi_df[mask]
ax.plot(phi_plot.index, phi_plot['phi_MV'], color='#1f77b4', linewidth=1.5)
ax.fill_between(phi_plot.index, 0, phi_plot['phi_MV'], alpha=0.2, color='#1f77b4')
ax.axvspan(pd.Timestamp('2011-11-26'), pd.Timestamp('2012-07-06'),
           alpha=0.3, color='orange', label='MSL transit (Zeitlin 2013)')
ax.set_ylabel('Modulation potential φ (MV)')
ax.set_title('Solar Modulation Potential φ(t) — Usoskin et al. Database')
ax.legend()
ax.set_ylim(0, 1500)

# Annotate solar min/max
ax.annotate('Solar min\n(cycle 24)', xy=(pd.Timestamp('2009-06-01'), 380),
            xytext=(pd.Timestamp('2007-01-01'), 550),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=9, ha='center')
ax.annotate('Solar max\n(cycle 24)', xy=(pd.Timestamp('2014-04-01'), 1100),
            xytext=(pd.Timestamp('2016-01-01'), 950),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=9, ha='center')

# Bottom: integrated proton flux >100 MeV/n vs phi
ax = axes[1]
E_int = np.logspace(2, 5, 100)  # 100 MeV to 100 GeV
phi_range = np.linspace(300, 1400, 100)
integrated_flux = [np.trapz(force_field_modulation(E_int, phi, lis_proton), E_int)
                   for phi in phi_range]
ax.plot(phi_range, integrated_flux, color='#d62728', linewidth=2)
ax.set_xlabel('Modulation potential φ (MV)')
ax.set_ylabel(r'Integrated proton flux $>$100 MeV (cm$^{-2}$ s$^{-1}$ sr$^{-1}$)')
ax.set_title('Integrated Proton Flux vs Solar Modulation')
ax.axvline(550, color='orange', linestyle='--', label='MSL cruise (φ≈550 MV)')
ax.legend()

plt.tight_layout()
plt.savefig('../figures/01_solar_cycle.png', dpi=150, bbox_inches='tight')
plt.show()

# Print MSL cruise phi statistics
msl_mask = (phi_df.index >= '2011-11-26') & (phi_df.index <= '2012-07-06')
phi_msl = phi_df[msl_mask]['phi_MV']
print(f'MSL transit φ: mean={phi_msl.mean():.0f} MV, '
      f'min={phi_msl.min():.0f} MV, max={phi_msl.max():.0f} MV')

## 4. Total GCR flux by species contribution

While HZE ions (C, O, Si, Fe) are a tiny fraction of the total particle flux,
they dominate dose **equivalent** because of Z² LET scaling and high quality factors Q(L).

In [ ]:
phi_test = 550.0  # MV (MSL cruise conditions)
flux_dict = gcr_total_flux(E, phi_test)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: flux by species
ax = axes[0]
for sp_key in ['H', 'He', 'C', 'O', 'Si', 'Fe']:
    sp = ION_SPECIES[sp_key]
    ax.loglog(E, flux_dict[sp_key], color=species_colors[sp_key],
              label=f"{sp_key} (Z={sp['Z']})", linewidth=2)
ax.loglog(E, flux_dict['total'], 'k-', linewidth=1.5, alpha=0.7, label='Total')
ax.set_xlabel('Kinetic energy (MeV/nucleon)')
ax.set_ylabel(r'Differential flux (cm$^{-2}$ s$^{-1}$ MeV$^{-1}$ sr$^{-1}$)')
ax.set_title(f'GCR Flux by Species (φ = {phi_test:.0f} MV)')
ax.legend(fontsize=9)
ax.set_xlim(10, 1e5)

# Right: flux fraction by species vs energy
ax = axes[1]
total_flux = flux_dict['total']
cumulative = np.zeros_like(E)
for sp_key in ['H', 'He', 'C', 'O', 'Si', 'Fe']:
    fraction = flux_dict[sp_key] / np.maximum(total_flux, 1e-30)
    ax.fill_between(E, cumulative, cumulative + fraction,
                    label=sp_key, color=species_colors[sp_key], alpha=0.85)
    cumulative += fraction

ax.set_xscale('log')
ax.set_xlabel('Kinetic energy (MeV/nucleon)')
ax.set_ylabel('Fraction of total flux')
ax.set_title('Species Fraction of Total GCR Flux')
ax.legend(loc='center right', fontsize=9)
ax.set_xlim(10, 1e5)
ax.set_ylim(0, 1)

plt.suptitle(f'Total GCR Flux at φ = {phi_test:.0f} MV', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../figures/01_flux_by_species.png', dpi=150, bbox_inches='tight')
plt.show()

# Integrated flux fractions
print(f'\nIntegrated flux fractions (10 MeV/n – 100 GeV/n, φ={phi_test:.0f} MV):')
total_int = np.trapz(flux_dict['total'], E)
for sp_key in ['H', 'He', 'C', 'O', 'Si', 'Fe']:
    sp_int = np.trapz(flux_dict[sp_key], E)
    print(f'  {sp_key:<4}: {sp_int/total_int*100:5.2f}% of total flux')

## 5. Impact of HZE spectral shape on flux at different energies

The key difference between abundance-scaled proton LIS (old) and Boschini per-species LIS (new):
Fe with α=2.65 (vs proton α=3.93) has significantly more flux above ~1 GeV/n.

In [ ]:
# Compare old (abundance-scaled) vs new (Boschini per-species) for Fe and C
fig, ax = plt.subplots(figsize=(9, 5))

# Old: abundance-scaled proton LIS
abundance_Fe = ION_SPECIES['Fe']['abundance']
abundance_C = ION_SPECIES['C']['abundance']
flux_Fe_old = abundance_Fe * lis_proton(E)
flux_C_old = abundance_C * lis_proton(E)

# New: Boschini per-species
flux_Fe_new = lis_heavy(E, 26, 56)
flux_C_new = lis_heavy(E, 6, 12)

ax.loglog(E, flux_Fe_old * E**2.7, 'b--', linewidth=1.5,
          label='Fe — abundance-scaled (old, α=3.93)', alpha=0.7)
ax.loglog(E, flux_Fe_new * E**2.7, 'b-', linewidth=2,
          label='Fe — Boschini (new, α=2.65)')
ax.loglog(E, flux_C_old * E**2.7, 'g--', linewidth=1.5,
          label='C — abundance-scaled (old, α=3.93)', alpha=0.7)
ax.loglog(E, flux_C_new * E**2.7, 'g-', linewidth=2,
          label='C — Boschini (new, α=2.77)')

ax.axvline(1000, color='gray', linestyle=':', alpha=0.7, label='Reference: 1 GeV/n')
ax.set_xlabel('Kinetic energy (MeV/nucleon)')
ax.set_ylabel(r'$j \times E^{2.7}$')
ax.set_title('HZE Spectra: Abundance-Scaled vs Boschini et al. (2020)\n'
             'Normalized to match at 1 GeV/n; spectra differ at other energies')
ax.legend(fontsize=9)
ax.set_xlim(10, 1e5)

plt.tight_layout()
plt.savefig('../figures/01_hze_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Ratio at high energies
E_check = np.array([100, 1000, 10000, 100000])
print('\nFe flux ratio (Boschini / old) at selected energies:')
for E_c in E_check:
    r_Fe = lis_heavy(np.array([E_c]), 26, 56)[0] / (abundance_Fe * lis_proton(np.array([E_c]))[0])
    r_C = lis_heavy(np.array([E_c]), 6, 12)[0] / (abundance_C * lis_proton(np.array([E_c]))[0])
    print(f'  E={E_c:>7.0f} MeV/n: Fe ratio={r_Fe:.2f}, C ratio={r_C:.2f}')